# Limpieza de Datos - Proyecto Banca
# Equipo 30 - 06/01/2026


In [1]:
import pandas as pd
import numpy as np

# Configuración de visualización
pd.set_option('display.max_columns', None)


---
## 1. Carga y Preparación Inicial
---

In [2]:
# Definición de rutas relativas
import os

# Base de la ruta (carpeta Data del proyecto)
# Desde Scripts/06-01-2026 subimos dos niveles a Equip_30
ruta_raw = os.path.join("..", "..", "Data", "06-01-2026", "06-01-2026_Raw.csv")

# Carga del dataset
df = pd.read_csv(ruta_raw)

print(f"Dimensiones iniciales: {df.shape}")


Dimensiones iniciales: (10987, 18)


---
## 2. Comprobacion de Duplicados
---

* 1. Comprobar duplicados por ID único
* 2. Comprobar si hay filas exactamente iguales (sin contar el ID)
* 3. Limpieza (en caso de que existan, elimina manteniendo la primera aparición)

In [3]:
duplicados_id = df.duplicated(subset=['id']).sum()
print(f"Filas con ID duplicado: {duplicados_id}")


columnas_sin_id = [col for col in df.columns if col != 'id']
duplicados_filas = df.duplicated(subset=columnas_sin_id).sum()
print(f"Filas duplicadas en contenido (ignorando ID): {duplicados_filas}")


if duplicados_filas > 0:
    df = df.drop_duplicates(subset=columnas_sin_id, keep='first')

Filas con ID duplicado: 0
Filas duplicadas en contenido (ignorando ID): 0


In [4]:
df.shape

(10987, 18)

In [5]:

# Eliminación de posibles duplicados
df = df.drop_duplicates()
print(f"Dimensiones tras eliminar duplicados: {df.shape}")


Dimensiones tras eliminar duplicados: (10987, 18)


---
## 3. Tratamiento de Valores Faltantes
---

* 1. Imputar numéricos ('age') con la mediana
* 2. Imputar categóricos ('marital', 'education', 'job') con la moda (para los nulls de marital y education, y los unknown de education y job)
* 3. Manejo de "unknown" estratégicos
    * 'contact' (20%) se quedan como "unknown" porque ya es una categoría.
    * Renombrar el "unknown" de 'poutcome' a su significado de negocio 'no_pcampaign'.

In [6]:
# Imputación de valores faltantes y renombrado de categorías

def get_mode(series):
    """Obtiene la moda gestionando grupos vacíos."""
    mode = series.mode()
    return mode[0] if not mode.empty else 'unknown'

# Grupos de edad temporales para imputación
bins = [17, 24, 34, 49, 64, 95]
labels = ['18-24', '25-34', '35-49', '50-64', '65+']
df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels, right=True)

# Imputación de Edad (age)
# Basado en educación y trabajo
df['age'] = df.groupby(['education', 'job'], observed=True)['age'].transform(
    lambda x: x.fillna(x.median())
)
df['age'] = df.groupby('job', observed=True)['age'].transform(
    lambda x: x.fillna(x.median())
)
df['age'] = df.groupby('education', observed=True)['age'].transform(
    lambda x: x.fillna(x.median())
)
df['age'] = df['age'].fillna(df['age'].median())

# Imputación de Educación (education)
# Basado en trabajo y grupo de edad
df['education'] = df.groupby(['job', 'age_group'], observed=True)['education']\
    .transform(lambda x: x.fillna(get_mode(x)))

df['education'] = df.groupby('job', observed=True)['education'].transform(
    lambda x: x.fillna(get_mode(x))
)
df['education'] = df.groupby('age_group', observed=True)['education'].transform(
    lambda x: x.fillna(get_mode(x))
)
df['education'] = df['education'].fillna(get_mode(df['education']))

# Imputación de Estado Civil (marital)
# Basado en grupo de edad y trabajo
df['marital'] = df.groupby(['age_group', 'job'], observed=True)['marital']\
    .transform(lambda x: x.fillna(get_mode(x)))

df['marital'] = df.groupby('age_group', observed=True)['marital'].transform(
    lambda x: x.fillna(get_mode(x))
)
df['marital'] = df.groupby('job', observed=True)['marital'].transform(
    lambda x: x.fillna(get_mode(x))
)
df['marital'] = df['marital'].fillna(get_mode(df['marital']))

# Renombrar 'unknown' en poutcome a 'no_pcampaign'
df['poutcome'] = df['poutcome'].replace('unknown', 'no_pcampaign')

# Eliminar columna auxiliar
df.drop('age_group', axis=1, inplace=True)

print("Nulos tras imputación:\n", df.isnull().sum())
print("\nValores únicos en poutcome:", df['poutcome'].unique())



Nulos tras imputación:
 id           0
age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
deposit      0
dtype: int64

Valores únicos en poutcome: <ArrowStringArray>
['no_pcampaign', 'other', 'failure', 'success']
Length: 4, dtype: str


---
## 4. Estandarización de formatos y tipos de datos
---

* 1. Estandarizar textos a minúsculas y sin espacios
* 2. Mapear todas las variables binarias de yes/no a 1/0 enteros (dejamos el dataset nativamente listo para cualquier algoritmo estadístico sin necesidad de hacer pasos extra más adelante.)
* 3. Corrección de tipos numéricos (asegurar enteros)
* 4. Convertir columnas categóricas multi-clase a tipo 'category' (las columnas de texto (object) consumen mucha memoria porque guardan cada palabra textualmente. El tipo category funciona como un diccionario: asigna un número interno (ej. Married = 1, Single = 2) pero nos sigue mostrando el texto)

In [7]:
columnas_texto = df.select_dtypes(include=['object']).columns
for col in columnas_texto:
    df[col] = df[col].astype(str).str.strip().str.lower()


columnas_binarias = ['deposit', 'default', 'housing', 'loan']
for col in columnas_binarias:
    df[col] = df[col].map({'yes': 1, 'no': 0})


df['age'] = df['age'].astype(int)

columnas_a_categoria = ['job', 'marital', 'education', 'contact', 'month', 'poutcome']
for col in columnas_a_categoria:
    if col in df.columns:
        df[col] = df[col].astype('category')

C:\Users\raari\AppData\Local\Temp\ipykernel_23964\1368230567.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columnas_texto = df.select_dtypes(include=['object']).columns


---
## 5.Verificación final del estado del Dataframe
---

In [8]:
print(f"Final Dataframe Shape: {df.shape}")
print("\nMissing values check:")
print(df.isnull().sum())
print("\nOptimized Data Types:")
print(df.dtypes)
print("\nUnique values in poutcome:")
print(df['poutcome'].unique())


Final Dataframe Shape: (10987, 18)

Missing values check:
id           0
age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
deposit      0
dtype: int64

Optimized Data Types:
id              int64
age             int64
job          category
marital      category
education    category
default         int64
balance         int64
housing         int64
loan            int64
contact      category
day             int64
month        category
duration        int64
campaign        int64
pdays           int64
previous        int64
poutcome     category
deposit         int64
dtype: object

Unique values in poutcome:
['no_pcampaign', 'other', 'failure', 'success']
Categories (4, str): ['failure', 'no_pcampaign', 'other', 'success']


In [9]:
# Guardado del dataset limpio
ruta_clean = ruta_raw.replace('Raw.csv', 'Clean.csv')
df.to_csv(ruta_clean, index=False)

print(f"\nArchivo guardado en: {ruta_clean}")


Archivo guardado en: ..\..\Data\06-01-2026\06-01-2026_Clean.csv
